# Lab 16: VEGAS Tutorial

In this lab, we will learn how to use the VEGAS algorithm to estimate the integral of a function. VEGAS is a Monte Carlo algorithm that uses importance sampling to estimate the integral of a function. The algorithm is particularly useful for high-dimensional integrals, where traditional methods such as the trapezoidal rule or Simpson's rule become impractical.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Two Gaussian distributions

We will attempt to produce 100,000 events of the following function composed of two Gaussian distributions:

$$\begin{aligned}f(x, y) = {} & A_1 \exp\left [-\frac{1}{2} \left ( \frac{(x - \mu_{1x})^2}{\sigma_{1x}^2} + \frac{(y - \mu_{1y})^2}{\sigma_{1y}^2} \right ) \right] \\
&+ A_2 \exp\left [-\frac{1}{2} \left ( \frac{(x - \mu_{2x})^2}{\sigma_{2x}^2} + \frac{(y - \mu_{2y})^2}{\sigma_{2y}^2} \right ) \right]\end{aligned}$$

## Parameters

over the range $0 < x, y < 5$. We will use the following parameters for the function:
$$A_1 = 1.0, \quad\mu_{1x} = 2.5, \quad\sigma_{1x} = 1.0, \quad\mu_{1y} = 2.5, \quad\sigma_{1y} = 1.0$$
$$A_2 = 4.0, \quad\mu_{2x} = 2.5, \quad\sigma_{2x} = 0.1, \quad\mu_{2y} = 2.5, \quad\sigma_{2y} = 0.1$$

In [ ]:
A1, A2 = 1.0, 4.0
MU1X = MU1Y = MU2X = MU2Y = 2.5
SIGMA1X = SIGMA1Y = 1.0
SIGMA2X = SIGMA2Y = 0.1
XMIN = YMIN = 0.0
XMAX = YMAX = 5.0

## References

1. T. Barklow, "Monte Carlo Techniques", https://courses.physics.illinois.edu/phys524/fa2023/topics.html (2023).
2. G. P. Lepage, "A New Algorithm for Adaptive Multidimensional Integration", J. Comput. Phys. 27, 192 (1978).
3. G. P. Lepage, "Adaptive multidimensional integration: VEGAS enhanced", J. Comput. Phys. 439, 110386 (2021).

## The function

In [ ]:
def components():
    return ((A1, MU1X, MU1Y, SIGMA1X, SIGMA1Y),
            (A2, MU2X, MU2Y, SIGMA2X, SIGMA2Y))

def f(x, y):
    return sum(a * np.exp(-0.5 * (((x-mx)/sx)**2 + ((y-my)/sy)**2))
               for a, mx, my, sx, sy in components())

## Lego plot

This lego plot will show the function we are trying to integrate.

In [ ]:
def lego_plot(xAmplitudes, yAmplitudes, nBins, xLabel, yLabel, title):
    hist, xe, ye = np.histogram2d(xAmplitudes, yAmplitudes, bins=nBins,
                                range=((XMIN,XMAX),(YMIN,YMAX)))
    xx, yy = np.meshgrid(xe[:-1], ye[:-1], indexing='ij')
    fig = plt.figure(figsize=(7,4.5))
    ax = fig.add_subplot(111, projection='3d')
    ax.bar3d(xx.ravel(), yy.ravel(), np.zeros(hist.size),
             np.diff(xe)[0], np.diff(ye)[0], hist.ravel(), color='#007c91')
    ax.set(xlabel=xLabel, ylabel=yLabel, zlabel='Events', title=title,
           xlim=(XMIN,XMAX), ylim=(YMIN,YMAX))
    plt.show()
    return ax

## Acceptance-rejection Method
Now, we define the standard acceptance-rejection method.

In [ ]:
def draw_proposals(xLow, yLow, size, rng):
    """Choose each interval equally often, then sample within its own width."""
    dx, dy = np.diff(xLow), np.diff(yLow)
    ix = rng.integers(len(dx), size=size)
    iy = rng.integers(len(dy), size=size)
    x = xLow[ix] + dx[ix] * rng.random(size)
    y = yLow[iy] + dy[iy] * rng.random(size)
    q = 1.0 / (len(dx) * len(dy) * dx[ix] * dy[iy])
    return x, y, q

## Rejection bound

In [ ]:
def rejection_bound(xLow, yLow):
    """Conservative global bound on f/q for this positive Gaussian mixture.

    Each component is largest at the point in a cell closest to its mean.
    The sum of those component maxima bounds the mixture throughout that cell.
    """
    upper = np.zeros((len(xLow)-1, len(yLow)-1))
    for a, mx, my, sx, sy in components():
        if a < 0 or sx <= 0 or sy <= 0:
            raise ValueError('Require nonnegative amplitudes and positive widths.')
        cx = np.clip(mx, xLow[:-1], xLow[1:])
        cy = np.clip(my, yLow[:-1], yLow[1:])
        upper += a*np.exp(-0.5*(((cx[:,None]-mx)/sx)**2 + ((cy[None,:]-my)/sy)**2))
    cell_inverse_q = (len(xLow)-1)*(len(yLow)-1)*np.outer(np.diff(xLow), np.diff(yLow))
    bound = float(np.max(upper * cell_inverse_q))
    if not np.isfinite(bound) or bound <= 0:
        raise ValueError('The rejection bound must be finite and positive.')
    return np.nextafter(bound, np.inf)

## Acceptance-rejection Method

In [ ]:
def sample_events(nPoints, xLow, yLow, seed=None):
    """Independent rejection sampling with a proved Gaussian-mixture envelope."""
    if nPoints < 1:
        raise ValueError('nPoints must be positive.')
    rng = np.random.default_rng(seed)
    bound = rejection_bound(xLow, yLow)
    accepted, proposals = [], []
    naccepted = neval = 0
    maximum = 0.0
    while naccepted < nPoints:
        x, y, q = draw_proposals(xLow, yLow, 4096, rng)
        weights = f(x, y)/q
        maximum = max(maximum, float(weights.max()))
        if np.any(weights > bound*(1+1e-12)):
            raise ValueError('Envelope violated; these samples are invalid.')
        keep = rng.random(len(x)) < weights/bound
        accepted.append(np.column_stack((x[keep], y[keep])))
        proposals.append(np.column_stack((x, y)))
        naccepted += int(keep.sum())
        neval += len(x)
    return {'events': np.concatenate(accepted)[:nPoints],
            'proposals': np.concatenate(proposals), 'nFunEval': neval,
            'nAccepted': naccepted, 'maxWeightEncountered': maximum, 'bound': bound}

## Uniform proposals

In [ ]:
def brute_force(nPoints, seed=None):
    result = sample_events(nPoints, np.array([XMIN, XMAX]), np.array([YMIN, YMAX]), seed)
    return {**result, 'yy1': result['events'][:,0], 'yy2': result['events'][:,1],
            'nTrainEval': 0, 'nTotalEval': result['nFunEval']}

## Vegas Method

In [ ]:
def rebin(edges, scores, alpha_damp):
    """Redistribute edges using the estimated importance of each interval."""
    z = scores / scores.sum()
    if np.any(z <= 0) or not np.all(np.isfinite(z)):
        raise ValueError('Adaptation requires finite positive bin scores.')
    if alpha_damp is not None:
        z = ((z - 1) / np.log(z)) ** alpha_damp
    cumulative = np.r_[0.0, np.cumsum(z)]
    new = np.interp(np.linspace(0, cumulative[-1], len(edges)), cumulative, edges)
    if np.any(np.diff(new) <= 0):
        raise ValueError('Adaptation produced a collapsed interval.')
    return new

## Vegas Method: intervals

In [ ]:
def setup_intervals(NN=40, KK=2000, nIterations=20, alpha_damp=1.5, seed=None):
    """Learn a product proposal; alpha_damp=None selects undamped allocation."""
    if NN < 2 or KK < NN or nIterations < 0:
        raise ValueError('Require NN >= 2, KK >= NN, and nIterations >= 0.')
    if not (XMIN < XMAX and YMIN < YMAX):
        raise ValueError('Each domain must have positive width.')
    if alpha_damp is not None and alpha_damp <= 0:
        raise ValueError('alpha_damp must be positive or None.')
    xLow = np.linspace(XMIN, XMAX, NN+1)
    yLow = np.linspace(YMIN, YMAX, NN+1)
    rng = np.random.default_rng(seed)
    for _ in range(nIterations):
        dx, dy = np.diff(xLow), np.diff(yLow)
        x, y, q = draw_proposals(xLow, yLow, KK, rng)
        ix = np.searchsorted(xLow, x, side='right') - 1
        iy = np.searchsorted(yLow, y, side='right') - 1
        nx, ny = np.bincount(ix, minlength=NN), np.bincount(iy, minlength=NN)
        if np.any(nx == 0) or np.any(ny == 0):
            raise ValueError('An adaptation bin is empty; increase KK.')
        values = f(x, y)
        px, py = 1/(NN*dx[ix]), 1/(NN*dy[iy])
        sx = dx * np.sqrt(np.bincount(ix, weights=(values/py)**2, minlength=NN)/nx)
        sy = dy * np.sqrt(np.bincount(iy, weights=(values/px)**2, minlength=NN)/ny)
        xLow, yLow = rebin(xLow, sx, alpha_damp), rebin(yLow, sy, alpha_damp)
    return xLow, yLow, np.diff(xLow), np.diff(yLow)

## Vegas Method: event generation

In [ ]:
def vegas(nPoints, NN=40, KK=2000, nIterations=20, alpha_damp=1.5, seed=None):
    train_seed, sample_seed = np.random.SeedSequence(seed).spawn(2)
    xLow, yLow, dx, dy = setup_intervals(NN, KK, nIterations, alpha_damp, train_seed)
    result = sample_events(nPoints, xLow, yLow, sample_seed)
    return {**result, 'yy1vegas': result['events'][:,0], 'yy2vegas': result['events'][:,1],
            'yy1vrho': result['proposals'][:,0], 'yy2vrho': result['proposals'][:,1],
            'vegasRatioMax': result['bound'], 'xLow': xLow, 'yLow': yLow,
            'nTrainEval': KK*nIterations, 'nTotalEval': KK*nIterations+result['nFunEval']}

## Compare the methods

Run both algorithms and compare results

In [ ]:
def plot_results(nPoints=100_000, nBins=40, NN=40, KK=2000,
                 nIterations=20, alpha_damp=1.5, seed=142):
    bf = brute_force(nPoints, seed)
    vg = vegas(nPoints, NN, KK, nIterations, alpha_damp, seed)
    lego_plot(bf['yy1'], bf['yy2'], nBins, 'x', 'y', 'Uniform rejection: accepted events')
    lego_plot(vg['yy1vrho'], vg['yy2vrho'], nBins, 'x', 'y', 'Adaptive proposals q(x,y)')
    lego_plot(vg['yy1vegas'], vg['yy2vegas'], nBins, 'x', 'y', 'Adaptive rejection: accepted events')
    for name, result in [('Uniform',bf), ('Adaptive',vg)]:
        print(f"{name}: training={result['nTrainEval']:,}; production={result['nFunEval']:,}; "
              f"total={result['nTotalEval']:,}; returned events={nPoints:,}")
        print(f"  maximum weight={result['maxWeightEncountered']:.6g}; "
              f"bound={result['bound']:.6g}")
    return bf, vg

In [ ]:
bf = brute_force(100_000, seed=142)
vg = vegas(100_000, seed=142)

## Acceptance-rejection Method

In [ ]:
lego_plot(bf['yy1'], bf['yy2'], 40, '$x$', '$y$',
          'Acceptance-rejection Method f(x,y)');

## Vegas Method: proposals

In [ ]:
lego_plot(vg['yy1vrho'], vg['yy2vrho'], 40, '$x$', '$y$',
          'Vegas Method q(x,y)');

## Vegas Method: accepted events

In [ ]:
lego_plot(vg['yy1vegas'], vg['yy2vegas'], 40, '$x$', '$y$',
          'Vegas Method f(x,y)');

## Function evaluations

In [ ]:
for name, result in [('Uniform', bf), ('Adaptive', vg)]:
    print(f"{name}: training={result['nTrainEval']:,}; "
          f"production={result['nFunEval']:,}; total={result['nTotalEval']:,}")
    print(f"  maximum weight={result['maxWeightEncountered']:.6g}; "
          f"bound={result['bound']:.6g}")

## Exercises

1. Try without the `alpha_damp` parameter, i.e. use the raw interval scores:

```python
plot_results(100_000, alpha_damp=None, seed=142)
```

This selects `z = scores / scores.sum()` without the damping transformation in `rebin`.

## Exercises

2. Change the form of the Gaussians to be offset from one another, e.g.

```python
A1 = A2 = 1.0
MU1X, MU1Y = 2.5, -2.5
MU2X, MU2Y = -2.5, 2.5
SIGMA1X = SIGMA1Y = 1.0
SIGMA2X = SIGMA2Y = 1.0
XMIN = YMIN = -5.0
XMAX = YMAX = 5.0
plot_results(100_000, seed=142)
```

The rejection bound is recomputed for these parameters.